<a href="https://colab.research.google.com/github/maddybfuturesfund/GooglePEGASUS_Dialogue_Summarization/blob/main/PEGASUS_Diaglogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q rouge_score bert_score evaluate

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU: NONE


In [ ]:
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, PegasusForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)

model_name = "google/pegasus-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

ds = load_dataset("knkarthick/samsum")
train_ds = ds["train"].shuffle(seed=42).select(range(2000))
val_ds   = ds["validation"].shuffle(seed=42).select(range(150))

def prep(batch):
    enc = tokenizer(batch["dialogue"], max_length=384, truncation=True)
    enc["labels"] = tokenizer(text_target=batch["summary"], max_length=64,
                              truncation=True)["input_ids"]
    return enc

tok_train = train_ds.map(prep, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(prep, batched=True, remove_columns=val_ds.column_names)

OUT = "/content/drive/MyDrive/pegasus_samsum"

args = Seq2SeqTrainingArguments(
    output_dir=OUT,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    max_steps=300,
    warmup_steps=20,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False, bf16=False,
    optim="adafactor",
    eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=100, save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    prediction_loss_only=True,
    logging_steps=10,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tok_train, eval_dataset=tok_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)
trainer.train()

model.save_pretrained(OUT + "/final")
tokenizer.save_pretrained(OUT + "/final")
print("saved to", OUT + "/final")


64 token output

In [ ]:
import json, gc, torch, evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, PegasusForConditionalGeneration

# Free the GPU if Cell 2 ran in this session
for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

MODEL_DIR = "/content/drive/MyDrive/pegasus_samsum/final"   # change if you unzipped the Kaggle model elsewhere
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
ds = load_dataset("knkarthick/samsum")
test_ds = ds["test"].shuffle(seed=42).select(range(200))    # same 200 as before

model = PegasusForConditionalGeneration.from_pretrained(MODEL_DIR).to(device).eval()
model.generation_config.max_length = None                   # silences the max_length warning

texts = list(test_ds["dialogue"])                           # list() avoids the Column error
refs  = list(test_ds["summary"])

summaries = []
for i in range(0, len(texts), 4):
    b = tokenizer(texts[i:i+4], return_tensors="pt", padding=True,
                  truncation=True, max_length=384).to(device)
    with torch.no_grad():
        out = model.generate(**b, num_beams=4, max_new_tokens=64)
    summaries += tokenizer.batch_decode(out, skip_special_tokens=True)
    del b, out
    torch.cuda.empty_cache()

del model; gc.collect(); torch.cuda.empty_cache()

# Save summaries first so a metrics crash can't lose them
json.dump({"summaries": summaries, "refs": refs},
          open("/content/drive/MyDrive/summaries.json", "w"))

rouge = evaluate.load("rouge")
rg = rouge.compute(predictions=summaries, references=refs)
print({"ROUGE-1": rg["rouge1"], "ROUGE-2": rg["rouge2"], "ROUGE-L": rg["rougeL"]})

bertscore = evaluate.load("bertscore")
bs = bertscore.compute(predictions=summaries, references=refs, lang="en")
mean = lambda x: sum(x) / len(x)
print({"BERTScore P": mean(bs["precision"]), "BERTScore R": mean(bs["recall"]),
       "BERTScore F1": mean(bs["f1"])})

for d, r, s in list(zip(texts, refs, summaries))[:3]:
    print(f"\nDIALOGUE:  {d[:200]}...\nREFERENCE: {r}\nGENERATED: {s}")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

{'ROUGE-1': np.float64(0.4513798990524955), 'ROUGE-2': np.float64(0.22128382126121043), 'ROUGE-L': np.float64(0.377192315283855)}


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'BERTScore P': 0.9249942913651467, 'BERTScore R': 0.9108925059437751, 'BERTScore F1': 0.9176513811945916}

DIALOGUE:  Claire: <file_photo>
Kim: Looks delicious...
Linda: No way... Look what I'm cooking right now:
Linda: <file_photo>
Claire: hahahaha 
Kim: Curry dream team
Claire: Enjoy your dinner :*...
REFERENCE: Both Claire and Linda are making curry for dinner. 
GENERATED: file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file

DIALOGUE:  Alyssa: Have you seen Fergie’s national anthem? Illuminati does a great job.
Derek: This is not normal. I saw it last week…
Alyssa: What do you think about it?
Derek: I can fart bright stripes and bri...
REFERENCE: Derek and Alyssa make fun of Fergie's performance of the national anthem.
GENERATED: Derek saw Fergie's national anthem last week.

DIALOGUE:  Ann: Hi, is the laptop still available?
Josh: Yes it is
Ann: I can pay 200 dollars
Josh: The price is 250 and it's non-negoti

128 output token

In [ ]:
import json, gc, torch, evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, PegasusForConditionalGeneration

# Free the GPU if Cell 2 ran in this session
for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

MODEL_DIR = "/content/drive/MyDrive/pegasus_samsum/final"   # change if you unzipped the Kaggle model elsewhere
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
ds = load_dataset("knkarthick/samsum")
test_ds = ds["test"].shuffle(seed=42).select(range(200))    # same 200 as before

model = PegasusForConditionalGeneration.from_pretrained(MODEL_DIR).to(device).eval()
model.generation_config.max_length = None                   # silences the max_length warning

texts = list(test_ds["dialogue"])                           # list() avoids the Column error
refs  = list(test_ds["summary"])

summaries = []
for i in range(0, len(texts), 4):
    b = tokenizer(texts[i:i+4], return_tensors="pt", padding=True,
                  truncation=True, max_length=384).to(device)
    with torch.no_grad():
        out = model.generate(**b, num_beams=4, max_new_tokens=128, length_penalty=2.0)
    summaries += tokenizer.batch_decode(out, skip_special_tokens=True)
    del b, out
    torch.cuda.empty_cache()

del model; gc.collect(); torch.cuda.empty_cache()

# Save summaries first so a metrics crash can't lose them
json.dump({"summaries": summaries, "refs": refs},
          open("/content/drive/MyDrive/summaries.json", "w"))

rouge = evaluate.load("rouge")
rg = rouge.compute(predictions=summaries, references=refs)
print({"ROUGE-1": rg["rouge1"], "ROUGE-2": rg["rouge2"], "ROUGE-L": rg["rougeL"]})

bertscore = evaluate.load("bertscore")
bs = bertscore.compute(predictions=summaries, references=refs, lang="en")
mean = lambda x: sum(x) / len(x)
print({"BERTScore P": mean(bs["precision"]), "BERTScore R": mean(bs["recall"]),
       "BERTScore F1": mean(bs["f1"])})

for d, r, s in list(zip(texts, refs, summaries))[:3]:
    print(f"\nDIALOGUE:  {d[:200]}...\nREFERENCE: {r}\nGENERATED: {s}")

{'ROUGE-1': np.float64(0.44999592464590943), 'ROUGE-2': np.float64(0.21686824165369425), 'ROUGE-L': np.float64(0.37395808015967935)}

* DIALOGUE:  Claire: <file_photo>
Kim: Looks delicious...
Linda: No way... Look what I'm cooking right now:
Linda: <file_photo>
Claire: hahahaha
Kim: Curry dream team
Claire: Enjoy your dinner :*...
REFERENCE: Both Claire and Linda are making curry for dinner.
GENERATED: file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file_photo> file

* DIALOGUE:  Alyssa: Have you seen Fergie’s national anthem? Illuminati does a great job.
Derek: This is not normal. I saw it last week…
Alyssa: What do you think about it?
Derek: I can fart bright stripes and bri...
REFERENCE: Derek and Alyssa make fun of Fergie's performance of the national anthem.
GENERATED: Derek and Alyssa saw Fergie's national anthem last week.

* DIALOGUE:  Ann: Hi, is the laptop still available?
Josh: Yes it is
Ann: I can pay 200 dollars
Josh: The price is 250 and it's non-negotiable
Ann: Do you have a bag for it? Some other accessories?
Josh: I have a ...
REFERENCE: Ann wants to buy Josh's laptop for $200. Josh doesn't want to negotiate the price. Ann will take it for $250 with accessories.
GENERATED: Josh has a bag and a small usb mouse for Ann to pick up.


# BASELINE

In [ ]:
import json, gc, torch, evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, PegasusForConditionalGeneration

gc.collect(); torch.cuda.empty_cache()

BASE = "google/pegasus-large"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(BASE)
ds = load_dataset("knkarthick/samsum")
test_ds = ds["test"].shuffle(seed=42).select(range(200))

model = PegasusForConditionalGeneration.from_pretrained(BASE).to(device).eval()
model.generation_config.max_length = None

texts = list(test_ds["dialogue"])
refs  = list(test_ds["summary"])

base_summaries = []
for i in range(0, len(texts), 4):
    b = tokenizer(texts[i:i+4], return_tensors="pt", padding=True,
                  truncation=True, max_length=384).to(device)
    with torch.no_grad():
        out = model.generate(**b, num_beams=4, max_new_tokens=64)
    base_summaries += tokenizer.batch_decode(out, skip_special_tokens=True)
    del b, out
    torch.cuda.empty_cache()

del model; gc.collect(); torch.cuda.empty_cache()

json.dump({"summaries": base_summaries, "refs": refs},
          open("/content/drive/MyDrive/baseline_summaries.json", "w"))

rouge = evaluate.load("rouge")
rg = rouge.compute(predictions=base_summaries, references=refs)
print({"ROUGE-1": rg["rouge1"], "ROUGE-2": rg["rouge2"], "ROUGE-L": rg["rougeL"]})

bertscore = evaluate.load("bertscore")
bs = bertscore.compute(predictions=base_summaries, references=refs, lang="en")
mean = lambda x: sum(x) / len(x)
print({"BERTScore P": mean(bs["precision"]), "BERTScore R": mean(bs["recall"]),
       "BERTScore F1": mean(bs["f1"])})

for d, r, s in list(zip(texts, refs, base_summaries))[:3]:
    print(f"\nDIALOGUE:  {d[:200]}...\nREFERENCE: {r}\nGENERATED: {s}")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'ROUGE-1': np.float64(0.2560760259392682), 'ROUGE-2': np.float64(0.056873496693116005), 'ROUGE-L': np.float64(0.19665305241556613)}


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'BERTScore P': 0.8430082941055298, 'BERTScore R': 0.8772457000613213, 'BERTScore F1': 0.8595733311772347}

DIALOGUE:  Claire: <file_photo>
Kim: Looks delicious...
Linda: No way... Look what I'm cooking right now:
Linda: <file_photo>
Claire: hahahaha 
Kim: Curry dream team
Claire: Enjoy your dinner :*...
REFERENCE: Both Claire and Linda are making curry for dinner. 
GENERATED: Look what I'm cooking right now: Linda: file_photo> Claire: hahahaha Kim: Curry dream team Claire: Enjoy your dinner :*

DIALOGUE:  Alyssa: Have you seen Fergie’s national anthem? Illuminati does a great job.
Derek: This is not normal. I saw it last week…
Alyssa: What do you think about it?
Derek: I can fart bright stripes and bri...
REFERENCE: Derek and Alyssa make fun of Fergie's performance of the national anthem.
GENERATED: Derek: This is not normal.

DIALOGUE:  Ann: Hi, is the laptop still available?
Josh: Yes it is
Ann: I can pay 200 dollars
Josh: The price is 250 and it's non-negotiable
Ann: Do you have a 

{'BERTScore P': 0.8430082878470421, 'BERTScore R': 0.877245699763298, 'BERTScore F1': 0.859573327600956}


* DIALOGUE:  Claire: <file_photo>
Kim: Looks delicious...
Linda: No way... Look what I'm cooking right now:
Linda: <file_photo>
Claire: hahahaha
Kim: Curry dream team
Claire: Enjoy your dinner :*...
REFERENCE: Both Claire and Linda are making curry for dinner.
GENERATED: Look what I'm cooking right now: Linda: file_photo> Claire: hahahaha Kim: Curry dream team Claire: Enjoy your dinner :*

* DIALOGUE:  Alyssa: Have you seen Fergie’s national anthem? Illuminati does a great job.
Derek: This is not normal. I saw it last week…
Alyssa: What do you think about it?
Derek: I can fart bright stripes and bri...
REFERENCE: Derek and Alyssa make fun of Fergie's performance of the national anthem.
GENERATED: Derek: This is not normal.

* DIALOGUE:  Ann: Hi, is the laptop still available?
Josh: Yes it is
Ann: I can pay 200 dollars
Josh: The price is 250 and it's non-negotiable
Ann: Do you have a bag for it? Some other accessories?
Josh: I have a ...
REFERENCE: Ann wants to buy Josh's laptop for $200. Josh doesn't want to negotiate the price. Ann will take it for $250 with accessories.
GENERATED: Josh: Yes it is Ann: I can pay 200 dollars Josh: The price is 250 and it's non-negotiable Ann: Do you have a bag for it?

In [ ]:
!pip install -q gradio

In [ ]:
import torch, gradio as gr
from transformers import AutoTokenizer, PegasusForConditionalGeneration

FINETUNED = "/content/drive/MyDrive/pegasus_samsum/final"
BASE = "google/pegasus-large"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

tokenizer = AutoTokenizer.from_pretrained(FINETUNED)

def load(path):
    m = PegasusForConditionalGeneration.from_pretrained(path).to(device).eval()
    m.generation_config.max_length = None
    return m

base_model = load(BASE)
tuned_model = load(FINETUNED)

def summarize(model, text):
    b = tokenizer([text], return_tensors="pt", truncation=True, max_length=384).to(device)
    with torch.no_grad():
        out = model.generate(**b, num_beams=4, max_new_tokens=64)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def compare(text):
    if not text or not text.strip():
        return "Please enter a dialogue.", "Please enter a dialogue."
    return summarize(base_model, text), summarize(tuned_model, text)

examples = [
    "Anna: Are we still on for dinner tonight?\nMark: Yes! I booked a table at 7.\nAnna: Great, I'll bring Sara.\nMark: Perfect, see you both there.",
    "Tom: Did you finish the report?\nLena: Not yet, I'm waiting for the sales numbers.\nTom: The deadline is tomorrow at noon.\nLena: I know, I'll send it tonight.",
]

with gr.Blocks(title="PEGASUS Dialogue Summarizer") as demo:
    gr.Markdown("# PEGASUS dialogue summarizer\nPaste a chat and compare the raw model with the model fine-tuned on SAMSum.")
    inp = gr.Textbox(label="Dialogue", lines=8,
                     placeholder="Name: message\nName: message\n...")
    btn = gr.Button("Summarize", variant="primary")
    with gr.Row():
        out_base = gr.Textbox(label="Baseline (google/pegasus-large)", lines=4)
        out_tuned = gr.Textbox(label="Fine-tuned (SAMSum)", lines=4)
    gr.Examples(examples=examples, inputs=inp)
    btn.click(compare, inputs=inp, outputs=[out_base, out_tuned])

demo.queue().launch(share=True, debug=True)

Running on: cpu


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea1a4acf1666816f14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import json, random, pandas as pd
from datasets import load_dataset

data = json.load(open("/content/drive/MyDrive/summaries.json"))
ds = load_dataset("knkarthick/samsum")
test_ds = ds["test"].shuffle(seed=42).select(range(200))
texts = list(test_ds["dialogue"])

random.seed(0)
idx = random.sample(range(len(texts)), 30)

rows = [{"i": i, "dialogue": texts[i], "reference": data["refs"][i],
         "generated": data["summaries"][i], "any_error": "", "notes": ""}
        for i in idx]
pd.DataFrame(rows).to_csv("/content/drive/MyDrive/error_analysis_longer.csv", index=False)